<a href="https://colab.research.google.com/github/Sernoth/generative-models-cats/blob/main/generative_cats_notebook_ddpm_patchdiffusion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Project III: Generative Models — Cat Image Generation

## 0. Google Drive · Install · Imports

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
DRIVE_DIR = '/content/drive/MyDrive/diffusion_cats'
os.makedirs(DRIVE_DIR, exist_ok=True)
print('Drive mounted →', DRIVE_DIR)

Mounted at /content/drive
Drive mounted → /content/drive/MyDrive/diffusion_cats


In [ ]:
!pip install -q torch torchvision diffusers accelerate datasets clean-fid torchmetrics opendatasets torch-fidelity torchcfm torchdiffeq

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.2/40.2 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 57.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.6/85.6 kB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.1/58.1 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 76.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.2/61.2 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 852.4/852.4 kB 66.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 125.5 MB/s eta 0:00:00


In [ ]:
import os, torch, numpy as np, itertools, json, pandas as pd
from pathlib import Path
from torchvision import transforms, utils
from torch.utils.data import DataLoader, Dataset
from torch import nn, optim
from tqdm.auto import tqdm
from PIL import Image
from cleanfid import fid as clean_fid
import os
import csv
import itertools
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from tqdm import tqdm
from torchdiffeq import odeint
from torch.nn.utils import clip_grad_norm_
from diffusers import UNet2DModel
from pathlib import Path
from torchcfm.conditional_flow_matching import ExactOptimalTransportConditionalFlowMatcher
DEVICE     = 'cuda' if torch.cuda.is_available() else 'cpu'
IMG_SIZE   = 64
BATCH      = 32
EPOCHS     = 50      # epochs per hparam combo
CKPT_EVERY = 10      # checkpoint cadence (epochs)
N_FAKE     = 1000
OUT_DIR    = Path('generated'); OUT_DIR.mkdir(exist_ok=True)
REAL_DIR   = Path('real_eval'); REAL_DIR.mkdir(exist_ok=True)

RESULTS    = {}      # (model__slug) -> {FID, KID}
print(DEVICE)

Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


cuda


## Helpers: checkpointing + scoring

**Checkpoint filename convention:** `{tag}__{hparam_slug}__ep{epoch:03d}.pt`  
e.g. `fastgan__nz256_ngf64_lr2e-4__ep010.pt`

In [ ]:
# ── checkpoint I/O ────────────────────────────────────────────────────────────
def _ckpt_path(tag, slug, epoch):
    return Path(DRIVE_DIR) / f'{tag}__{slug}__ep{epoch:03d}.pt'

def save_ckpt(tag, slug, epoch, **state):
    p = _ckpt_path(tag, slug, epoch)
    torch.save({'epoch': epoch, **state}, p)
    print(f'  [ckpt] {p.name}')

def load_ckpt(tag, slug):
    matches = sorted(Path(DRIVE_DIR).glob(f'{tag}__{slug}__ep*.pt'))
    if not matches:
        print(f'  [ckpt] no checkpoint for {tag} ({slug}) — starting fresh'); return None, 0
    state = torch.load(matches[-1], map_location=DEVICE)
    epoch = state.pop('epoch')
    print(f'  [ckpt] resumed {matches[-1].name} (ep {epoch})'); return state, epoch

# ── FID / KID scoring ─────────────────────────────────────────────────────────
def score(gen_dir: str, tag: str, slug: str):
    key = f'{tag}__{slug}'
    RESULTS[key] = {
        'FID': round(clean_fid.compute_fid(str(REAL_DIR), gen_dir, verbose = False), 2),
        'KID': round(clean_fid.compute_kid(str(REAL_DIR), gen_dir, verbose = False), 4),
    }
    print(f'  [score] {key}: {RESULTS[key]}')
    return RESULTS[key]

def save_imgs(tensors, directory):
    Path(directory).mkdir(parents=True, exist_ok=True)
    existing = len(list(Path(directory).glob('*.png')))
    for i, t in enumerate(tensors):
        utils.save_image((t.clamp(-1,1)+1)/2, f'{directory}/{existing+i}.png')

def build_real_ref(loader, n=1000):
    if len(list(REAL_DIR.glob('*.png'))) >= n: return
    print('Building real reference set...')
    count = 0
    for imgs in loader:
        for img in imgs:
            utils.save_image((img+1)/2, REAL_DIR/f'{count}.png'); count += 1
            if count >= n: return

## 1. Data

In [ ]:
import opendatasets as od
od.download('https://www.kaggle.com/datasets/crawford/cat-dataset')
CAT_DIR = 'cat-dataset'


Please provide your Kaggle credentials to download this dataset. Learn more: http://bit.ly/kaggle-creds
Your Kaggle username: apollosatori
Your Kaggle Key: ··········
Dataset URL: https://www.kaggle.com/datasets/crawford/cat-dataset


100%|██████████| 4.04G/4.04G [03:07<00:00, 23.1MB/s]


In [ ]:
tf = transforms.Compose([
    transforms.Resize(IMG_SIZE), transforms.CenterCrop(IMG_SIZE),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(), transforms.Normalize([.5]*3, [.5]*3),
])

class FlatImageDataset(Dataset):
    def __init__(self, root, transform=None):
        self.files = list(Path(root).rglob('*.jpg')) + list(Path(root).rglob('*.png'))
        self.transform = transform
    def __len__(self): return len(self.files)
    def __getitem__(self, i):
        img = Image.open(self.files[i]).convert('RGB')
        return self.transform(img) if self.transform else img

dataset = FlatImageDataset(CAT_DIR, tf)
loader  = DataLoader(dataset, batch_size=BATCH, shuffle=True, num_workers=2, pin_memory=True)
build_real_ref(loader)
CFM_BATCH  = 64
cfm_loader = DataLoader(dataset, batch_size=CFM_BATCH, shuffle=True,
                        num_workers=2, pin_memory=True)
print(f'{len(dataset)} training images, {len(list(REAL_DIR.glob("*.png")))} real reference images')

Building real reference set...
19994 training images, 1000 real reference images


## 3. PatchDiffusion — grid search

**Paper defaults (Karras et al. 2022, Table 1):** `σ_min=0.002, σ_max=80, P_mean=-1.2, P_std=1.2`  
These are fixed as they are the empirically established best values across datasets.  
Inference uses the **Heun sampler** (2nd-order Runge-Kutta) as specified in the paper.  
**Grid:** patch size × P_mean (subtle variation around paper default) × learning rate

In [ ]:
from diffusers import UNet2DModel, DDPMScheduler
CSV_OUTPUT_PATH = os.path.join(DRIVE_DIR, 'patchdiffusion_results.csv')
SIGMA_MIN, SIGMA_MAX = 0.002, 80.0
P_STD = 1.2  # fixed

DIFF_GRID = dict(
    patch_size = [16, 32],
    p_mean     = [-1.2, -0.6],
    lr         = [1e-4, 5e-5],
)
def heun_sample(unet, n_steps=35, batch_size=16):
    rho  = 7.0
    sigs = (SIGMA_MAX**(1/rho) + torch.linspace(0, 1, n_steps, device=DEVICE) *
            (SIGMA_MIN**(1/rho) - SIGMA_MAX**(1/rho))) ** rho
    sigs = torch.cat([sigs, sigs.new_zeros(1)])  # dopisanie 0 na końcu

    x = torch.randn(batch_size, 3, IMG_SIZE, IMG_SIZE, device=DEVICE) * sigs[0]
    unet.eval()
    with torch.no_grad():
        for i in range(len(sigs) - 1):
            s, s_next = sigs[i], sigs[i+1]
            t_idx = ((s / SIGMA_MAX) * 999).long().expand(batch_size)
            d     = (x - unet(x, t_idx).sample) / s          # kierunek score
            x_2   = x + (s_next - s) * d                     # krok Eulera
            if s_next > 0:
                t2  = ((s_next / SIGMA_MAX) * 999).long().expand(batch_size)
                d2  = (x_2 - unet(x_2, t2).sample) / s_next  # druga pochodna
                x   = x + (s_next - s) * (d + d2) / 2        # korekta Heuna
            else:
                x = x_2
    return x.clamp(-1, 1)


for patch_size, p_mean, lr in itertools.product(*DIFF_GRID.values()):
    slug = f'patch{patch_size}_pmean{p_mean}_lr{lr:.0e}'


    unet = UNet2DModel(
        sample_size=patch_size, in_channels=3, out_channels=3,
        layers_per_block=2, block_out_channels=(64, 128, 256),
        down_block_types=("DownBlock2D", "AttnDownBlock2D", "AttnDownBlock2D"),
        up_block_types=("AttnUpBlock2D", "AttnUpBlock2D", "UpBlock2D"),
    ).to(DEVICE)

    scheduler = DDPMScheduler(num_train_timesteps=1000, beta_schedule='squaredcos_cap_v2')
    opt       = optim.AdamW(unet.parameters(), lr=lr)

    state, start_ep = load_ckpt('patchdiff', slug)
    if state:
        unet.load_state_dict(state['unet'])
        opt.load_state_dict(state['opt'])
        print(f'[WCHODZENIE] Wznowiono trening od epoki: {start_ep}')

    for epoch in tqdm(range(start_ep, EPOCHS), desc=slug, leave=False):
        unet.train()
        for imgs in loader:
            imgs    = imgs.to(DEVICE)
            patches = transforms.RandomCrop(patch_size)(imgs)
            noise   = torch.randn_like(patches)

            # Log-normal noise level sampling (Karras et al. 2022)
            log_sig = torch.randn(patches.size(0), device=DEVICE) * P_STD + p_mean
            t       = (log_sig.exp().clamp(SIGMA_MIN, SIGMA_MAX) / SIGMA_MAX * 999).long()

            loss    = nn.functional.mse_loss(
                unet(scheduler.add_noise(patches, noise, t), t).sample, noise)

            opt.zero_grad()
            loss.backward()
            opt.step()

        current_epoch = epoch + 1
        if current_epoch % CKPT_EVERY == 0 or epoch == EPOCHS - 1:

            # A. Zapis wag modelu (.pt)
            save_ckpt('patchdiff', slug, current_epoch, unet=unet.state_dict(), opt=opt.state_dict())

            gen_dir = str(OUT_DIR / f'patchdiff_{slug}_ep{current_epoch:03d}')
            Path(gen_dir).mkdir(exist_ok=True)

            print(f'\n[Inference] generating {N_FAKE} images for epoch {current_epoch}...')
            for _ in range(0, N_FAKE, 16):
                save_imgs(heun_sample(unet, n_steps=35, batch_size=16).cpu(), gen_dir)

            metrics = score(gen_dir, 'patchdiff', slug)
            row = {
                'model_name': f'patchdiff_{slug}',
                'patch_size': patch_size,
                'p_mean': p_mean,
                'lr': lr,
                'epoch': current_epoch,
                'FID': metrics.get('FID') if metrics else None,
                'KID': metrics.get('KID') if metrics else None
            }

            if os.path.exists(CSV_OUTPUT_PATH):
                existing_df = pd.read_csv(CSV_OUTPUT_PATH)
                duplicate_mask = (existing_df['model_name'] == row['model_name']) & (existing_df['epoch'] == row['epoch'])
                existing_df = existing_df[~duplicate_mask]

                # Dołączamy nowy wiersz na koniec tabeli
                new_row_df = pd.DataFrame([row])
                updated_df = pd.concat([existing_df, new_row_df], ignore_index=True)
                updated_df.to_csv(CSV_OUTPUT_PATH, index=False)
            else:
                new_df = pd.DataFrame([row])
                new_df.to_csv(CSV_OUTPUT_PATH, index=False)

            print(f"  [save CSV] Wyniki modelu {row['model_name']} z epoki {current_epoch} zostały zsynchronizowane z Google Drive.")



Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.



=== URUCHAMIANIE: PatchDiffusion | patch16_pmean-1.2_lr1e-04 ===
  [ckpt] resumed patchdiff__patch16_pmean-1.2_lr1e-04__ep050.pt (ep 50)
[WCHODZENIE] Wznowiono trening od epoki: 50


patch16_pmean-1.2_lr1e-04: 0it [00:00, ?it/s]


=== URUCHAMIANIE: PatchDiffusion | patch16_pmean-1.2_lr5e-05 ===
  [ckpt] resumed patchdiff__patch16_pmean-1.2_lr5e-05__ep050.pt (ep 50)
[WCHODZENIE] Wznowiono trening od epoki: 50


patch16_pmean-1.2_lr5e-05: 0it [00:00, ?it/s]

...

patch32_pmean-1.2_lr5e-05: 0it [00:00, ?it/s]


=== URUCHAMIANIE: PatchDiffusion | patch32_pmean-0.6_lr1e-04 ===
  [ckpt] resumed patchdiff__patch32_pmean-0.6_lr1e-04__ep050.pt (ep 50)
[WCHODZENIE] Wznowiono trening od epoki: 50


patch32_pmean-0.6_lr1e-04: 0it [00:00, ?it/s]


=== URUCHAMIANIE: PatchDiffusion | patch32_pmean-0.6_lr5e-05 ===
  [ckpt] resumed patchdiff__patch32_pmean-0.6_lr5e-05__ep050.pt (ep 50)
[WCHODZENIE] Wznowiono trening od epoki: 50


patch32_pmean-0.6_lr5e-05: 0it [00:00, ?it/s]

## 4. WAE-MMD — grid search

**Paper defaults (Tolstikhin et al. 2019, official code):** `dz=64, λ=100, lr=1e-3`  
λ=100 is the CelebA default in the official repo — NOT λ=10 as is sometimes assumed.  
**Grid:** latent dimension × λ × learning rate

In [ ]:
from torchvision.models import resnet18

WAE_GRID = dict(
    dz  = [64, 128],
    lam = [10.0, 100.0],
    lr  = [1e-3, 3e-4],
)

def mmd_rbf(x, y, sigma=1.0):
    """Unbiased RBF-kernel MMD (Tolstikhin et al. 2019, Eq. 4)."""
    def K(a, b): return torch.exp(-torch.cdist(a,b)**2 / (2*sigma**2))
    return K(x,x).mean() + K(y,y).mean() - 2*K(x,y).mean()

for dz, lam, lr in itertools.product(*WAE_GRID.values()):
    slug = f'dz{dz}_lam{lam:.0f}_lr{lr:.0e}'
    print(f'\n=== WAE-MMD | {slug} ===')

    class Encoder(nn.Module):
        def __init__(self):
            super().__init__()
            base = resnet18()
            self.net = nn.Sequential(*list(base.children())[:-1], nn.Flatten(), nn.Linear(512, dz))
        def forward(self, x): return self.net(x)

    class Decoder(nn.Module):
        def __init__(self):
            super().__init__()
            self.net = nn.Sequential(
                nn.Linear(dz, 512*4*4), nn.Unflatten(1,(512,4,4)),
                *[nn.Sequential(nn.Upsample(scale_factor=2), nn.Conv2d(ci,co,3,1,1), nn.ReLU())
                  for ci,co in [(512,256),(256,128),(128,64),(64,32)]],
                nn.Upsample(size=IMG_SIZE), nn.Conv2d(32,3,3,1,1), nn.Tanh(),
            )
        def forward(self, z): return self.net(z)

    enc, dec = Encoder().to(DEVICE), Decoder().to(DEVICE)
    opt = optim.Adam(list(enc.parameters())+list(dec.parameters()), lr=lr)

    state, start_ep = load_ckpt('waemmd', slug)
    if state:
        enc.load_state_dict(state['enc']); dec.load_state_dict(state['dec'])
        opt.load_state_dict(state['opt'])

    for epoch in tqdm(range(start_ep, EPOCHS), desc=slug, leave=False):
        for imgs in loader:
            imgs = imgs.to(DEVICE); z = enc(imgs)
            loss = nn.functional.mse_loss(dec(z), imgs) + lam * mmd_rbf(z, torch.randn_like(z))
            opt.zero_grad(); loss.backward(); opt.step()

        if (epoch+1) % CKPT_EVERY == 0 or epoch == EPOCHS-1:
            save_ckpt('waemmd', slug, epoch+1,
                      enc=enc.state_dict(), dec=dec.state_dict(), opt=opt.state_dict())

    dec.eval(); gen_dir = str(OUT_DIR/f'waemmd_{slug}')
    with torch.no_grad():
        for _ in range(0, N_FAKE, BATCH):
            save_imgs(dec(torch.randn(BATCH, dz, device=DEVICE)).cpu(), gen_dir)
    score(gen_dir, 'waemmd', slug)

## 5. OT-CFM — grid search

**Paper defaults (Tong et al. 2024, torchcfm examples):** `sigma=0.0, lr=2e-4`  
`sigma=0.0` means fully deterministic conditional paths (OT-CFM definition).  
Small `sigma>0` adds Gaussian smoothing; the paper uses 0.0 as the canonical OT-CFM variant.  
**Grid:** sigma × U-Net channel configuration × learning rate

In [ ]:
CSV_OTCFM_PATH = os.path.join(DRIVE_DIR, 'ot_cfm_results.csv')

CFM_GRID = dict(
    sigma         = [0.0, 0.01],
    unet_channels = [
        (128, 256, 256, 256),
        (64,  128, 128, 128),
    ],
    lr            = [2e-4, 1e-4],
)

EMA_DECAY   = 0.9999
DROPOUT     = 0.1
GRAD_CLIP   = 1.0
CKPT_EVERY  = 10

def update_ema(ema_model, model, decay):
    """EMA update — ema_model stays on CPU, model is on GPU."""
    with torch.no_grad():
        for ema_p, p in zip(ema_model.parameters(), model.parameters()):
            ema_p.mul_(decay).add_(p.cpu(), alpha=1 - decay)

for sigma, unet_channels, lr in itertools.product(*CFM_GRID.values()):
    ch_str = '_'.join(str(c) for c in unet_channels)
    slug   = f'sig{sigma}_unet{ch_str}_lr{lr:.0e}'
    print(f'\n=== OT-CFM | {slug} ===')

    num_blocks  = len(unet_channels)
    down_blocks = ("DownBlock2D",) + ("AttnDownBlock2D",) * (num_blocks - 1)
    up_blocks   = ("AttnUpBlock2D",) * (num_blocks - 1) + ("UpBlock2D",)

    def make_unet():
        return UNet2DModel(
            sample_size=IMG_SIZE, in_channels=3, out_channels=3,
            layers_per_block=2, block_out_channels=unet_channels,
            down_block_types=down_blocks, up_block_types=up_blocks,
            dropout=DROPOUT,
        )

    fm_unet  = make_unet().to(DEVICE)

    ema_unet = make_unet().cpu()
    ema_unet.load_state_dict(fm_unet.state_dict())
    ema_unet.requires_grad_(False)

    cfm = ExactOptimalTransportConditionalFlowMatcher(sigma=sigma)
    opt = optim.Adam(fm_unet.parameters(), lr=lr, betas=(0.9, 0.999))

    state, start_ep = load_ckpt('otcfm', slug)
    if state:
        fm_unet.load_state_dict(state['unet'])
        ema_unet.load_state_dict(state['ema'])
        opt.load_state_dict(state['opt'])
        print(f'[WCHODZENIE] Wznowiono trening OT-CFM od epoki: {start_ep}')

    for epoch in tqdm(range(start_ep, EPOCHS), desc=slug, leave=False):
        fm_unet.train()
        for x1 in cfm_loader:
            x1 = x1.to(DEVICE)
            t, xt, ut = cfm.sample_location_and_conditional_flow(torch.randn_like(x1), x1)
            loss = nn.functional.mse_loss(fm_unet(xt, (t * 999).long()).sample, ut)
            opt.zero_grad()
            loss.backward()
            clip_grad_norm_(fm_unet.parameters(), GRAD_CLIP)
            opt.step()
            update_ema(ema_unet, fm_unet, EMA_DECAY)

        current_epoch = epoch + 1
        if current_epoch % CKPT_EVERY == 0 or epoch == EPOCHS - 1:

            # A. Zapis wag
            save_ckpt('otcfm', slug, current_epoch,
                      unet=fm_unet.state_dict(),
                      ema=ema_unet.state_dict(),  # CPU state_dict — torch.save obsługuje normalnie
                      opt=opt.state_dict())

            ema_unet.to(DEVICE).eval()
            gen_dir = str(OUT_DIR / f'otcfm_{slug}_ep{current_epoch:03d}')
            Path(gen_dir).mkdir(exist_ok=True)
            print(f'\n[Inference] Generowanie {N_FAKE} obrazków dla epoki {current_epoch}...')
            with torch.no_grad():
                for _ in range(0, N_FAKE, CFM_BATCH):
                    x0 = torch.randn(CFM_BATCH, 3, IMG_SIZE, IMG_SIZE, device=DEVICE)
                    def vf(t, x): return ema_unet(x, (t.expand(x.size(0)) * 999).long()).sample
                    samples = odeint(vf, x0, torch.linspace(0, 1, 20, device=DEVICE), method='euler')[-1]
                    save_imgs(samples.cpu(), gen_dir)
            ema_unet.cpu()
            torch.cuda.empty_cache()

            # C. Scoring + CSV
            metrics = score(gen_dir, 'otcfm', slug)
            row = {
                'model_name': f'otcfm_{slug}',
                'sigma':        sigma,
                'unet_channels': ch_str,
                'lr':           lr,
                'epoch':        current_epoch,
                'FID':          metrics.get('FID') if metrics else None,
                'KID':          metrics.get('KID') if metrics else None,
            }

            if os.path.exists(CSV_OTCFM_PATH):
                df_existing = pd.read_csv(CSV_OTCFM_PATH)
                mask = (df_existing['model_name'] == row['model_name']) & \
                       (df_existing['epoch']      == row['epoch'])
                df_existing = df_existing[~mask]
                pd.concat([df_existing, pd.DataFrame([row])], ignore_index=True) \
                  .to_csv(CSV_OTCFM_PATH, index=False)
            else:
                pd.DataFrame([row]).to_csv(CSV_OTCFM_PATH, index=False)

            print(f"  [ZAPIS CSV] {row['model_name']} ep{current_epoch} → {CSV_OTCFM_PATH}")

    # Zwolnij pamięć GPU po zakończeniu kombinacji
    del fm_unet, ema_unet, opt, cfm
    torch.cuda.empty_cache()


=== OT-CFM | sig0.0_unet128_256_256_256_lr2e-04 ===
  [ckpt] resumed otcfm__sig0.0_unet128_256_256_256_lr2e-04__ep050.pt (ep 50)
[WCHODZENIE] Wznowiono trening OT-CFM od epoki: 50



=== OT-CFM | sig0.0_unet128_256_256_256_lr1e-04 ===


  [ckpt] resumed otcfm__sig0.0_unet128_256_256_256_lr1e-04__ep050.pt (ep 50)
[WCHODZENIE] Wznowiono trening OT-CFM od epoki: 50



=== OT-CFM | sig0.0_unet64_128_128_128_lr2e-04 ===


  [ckpt] resumed otcfm__sig0.0_unet64_128_128_128_lr2e-04__ep050.pt (ep 50)
[WCHODZENIE] Wznowiono trening OT-CFM od epoki: 50


...

  [ckpt] resumed otcfm__sig0.01_unet128_256_256_256_lr1e-04__ep050.pt (ep 50)
[WCHODZENIE] Wznowiono trening OT-CFM od epoki: 50



=== OT-CFM | sig0.01_unet64_128_128_128_lr2e-04 ===


  [ckpt] resumed otcfm__sig0.01_unet64_128_128_128_lr2e-04__ep050.pt (ep 50)
[WCHODZENIE] Wznowiono trening OT-CFM od epoki: 50



=== OT-CFM | sig0.01_unet64_128_128_128_lr1e-04 ===


  [ckpt] resumed otcfm__sig0.01_unet64_128_128_128_lr1e-04__ep050.pt (ep 50)
[WCHODZENIE] Wznowiono trening OT-CFM od epoki: 50


In [ ]:
# 6. Full training

In [ ]:
# Patch Diffusion full training

In [ ]:

PATCH_SIZE   = 32
P_FULL       = 0.5
P_STD_SCHED  = [
    (1.0,  P_FULL),
    (0.5,  0.6 * (1 - P_FULL)),
    (0.25, 0.4 * (1 - P_FULL)),
]
BATCH_DIFF      = 64
LR_DIFF         = 2e-4
EPOCHS_DIFF     = 1600
CKPT_EVERY_DIFF = 200
N_STEPS_DDIM    = 50

SLUG_DIFF       = f'FULL_v2_patch{PATCH_SIZE}_pfull{P_FULL}'  # v2 — nowy slug
FINAL_DIFF_BASE = Path(DRIVE_DIR) / 'final_patchdiff_v2'
FINAL_DIFF_BASE.mkdir(parents=True, exist_ok=True)
csv_path_diff   = FINAL_DIFF_BASE / 'metrics_history.csv'

diff_loader = DataLoader(dataset, batch_size=BATCH_DIFF, shuffle=True,
                         num_workers=2, pin_memory=True)

# ── UNet ──────────────────────────────────────────────────────────────
unet_diff = UNet2DModel(
    sample_size=PATCH_SIZE,
    in_channels=5,       # 3 RGB + 2 coordinate channels
    out_channels=3,
    layers_per_block=2,
    block_out_channels=(128, 256, 256),
    down_block_types=("DownBlock2D", "AttnDownBlock2D", "AttnDownBlock2D"),
    up_block_types=("AttnUpBlock2D", "AttnUpBlock2D", "UpBlock2D"),
).to(DEVICE)

# POPRAWKA: używamy jednego schedulera dla treningu i inferencji
# Nie mieszamy EDM sigma schedule z DDPM add_noise
scheduler_diff = DDPMScheduler(
    num_train_timesteps=1000,
    beta_schedule='squaredcos_cap_v2',
)
opt_diff = optim.AdamW(unet_diff.parameters(), lr=LR_DIFF)

# ── coordinate channels ───────────────────────────────────────────────
def get_patch_with_coords(imgs, patch_h, patch_w):
    B, C, H, W = imgs.shape
    top  = torch.randint(0, H - patch_h + 1, (1,)).item()
    left = torch.randint(0, W - patch_w + 1, (1,)).item()
    patch = imgs[:, :, top:top+patch_h, left:left+patch_w]

    y_start = top  / (H - 1) * 2 - 1
    x_start = left / (W - 1) * 2 - 1
    y_end   = (top  + patch_h - 1) / (H - 1) * 2 - 1
    x_end   = (left + patch_w - 1) / (W - 1) * 2 - 1

    y = torch.linspace(y_start, y_end, patch_h, device=imgs.device)
    x = torch.linspace(x_start, x_end, patch_w, device=imgs.device)
    yy, xx = torch.meshgrid(y, x, indexing='ij')
    coords = torch.stack([yy, xx]).unsqueeze(0).expand(B, -1, -1, -1)
    return torch.cat([patch, coords], dim=1)   # (B, 5, ph, pw)

# ── DDIM sampler — spójny z treningiem DDPM ───────────────────────────
def ddim_sample(unet, batch_size=16, n_steps=50):

    sampler = DDIMScheduler(
        num_train_timesteps=1000,
        beta_schedule='squaredcos_cap_v2',
    )
    sampler.set_timesteps(n_steps)

    # Pełne współrzędne dla całego obrazu
    y = torch.linspace(-1, 1, IMG_SIZE, device=DEVICE)
    x = torch.linspace(-1, 1, IMG_SIZE, device=DEVICE)
    yy, xx = torch.meshgrid(y, x, indexing='ij')
    coords = torch.stack([yy, xx]).unsqueeze(0).expand(batch_size, -1, -1, -1)

    xt = torch.randn(batch_size, 3, IMG_SIZE, IMG_SIZE, device=DEVICE)

    unet.eval()
    with torch.no_grad():
        for t_step in sampler.timesteps:
            t_batch   = t_step.expand(batch_size).to(DEVICE)
            xt_input  = torch.cat([xt, coords], dim=1)   # (B, 5, H, W)
            pred_noise = unet(xt_input, t_batch).sample   # (B, 3, H, W)
            xt = sampler.step(pred_noise, t_step, xt).prev_sample

    return xt.clamp(-1, 1)

def ddim_sample_from_z(unet, z_noise, n_steps=50):
    """DDIM od konkretnego tensora szumu z — do interpolacji."""
    sampler = DDIMScheduler(
        num_train_timesteps=1000,
        beta_schedule='squaredcos_cap_v2',
    )
    sampler.set_timesteps(n_steps)

    B = z_noise.size(0)
    y = torch.linspace(-1, 1, IMG_SIZE, device=DEVICE)
    x = torch.linspace(-1, 1, IMG_SIZE, device=DEVICE)
    yy, xx = torch.meshgrid(y, x, indexing='ij')
    coords = torch.stack([yy, xx]).unsqueeze(0).expand(B, -1, -1, -1)

    xt = z_noise.to(DEVICE)

    unet.eval()
    with torch.no_grad():
        for t_step in sampler.timesteps:
            t_batch    = t_step.expand(B).to(DEVICE)
            xt_input   = torch.cat([xt, coords], dim=1)
            pred_noise = unet(xt_input, t_batch).sample
            xt = sampler.step(pred_noise, t_step, xt).prev_sample

    return xt.clamp(-1, 1)

# ── resume ────────────────────────────────────────────────────────────
state, start_ep = load_ckpt('patchdiff', SLUG_DIFF)
if state:
    unet_diff.load_state_dict(state['unet'])
    opt_diff.load_state_dict(state['opt'])
    print(f'[resume] od epoki {start_ep}')

# ── trening ───────────────────────────────────────────────────────────
scales, probs = zip(*P_STD_SCHED)

for epoch in tqdm(range(start_ep, EPOCHS_DIFF), desc='PatchDiffusion v2'):
    unet_diff.train()
    epoch_loss = 0.0

    for imgs in diff_loader:
        imgs = imgs.to(DEVICE)
        B    = imgs.size(0)

        scale = float(torch.tensor(scales)[
            torch.multinomial(torch.tensor(probs, dtype=torch.float), 1)
        ])
        ph = pw = IMG_SIZE if scale == 1.0 else int(IMG_SIZE * scale)

        inp = get_patch_with_coords(imgs, ph, pw)   # (B, 5, ph, pw)

        #jednostajne próbkowanie t — spójne z DDPMScheduler
        t     = torch.randint(0, 1000, (B,), device=DEVICE)
        noise = torch.randn(B, 3, ph, pw, device=DEVICE)
        noisy = scheduler_diff.add_noise(inp[:, :3], noise, t)

        noisy_with_coords = torch.cat([noisy, inp[:, 3:]], dim=1)  # (B,5,ph,pw)
        pred  = unet_diff(noisy_with_coords, t).sample              # (B,3,ph,pw)
        loss  = nn.functional.mse_loss(pred, noise)

        opt_diff.zero_grad(); loss.backward(); opt_diff.step()
        epoch_loss += loss.item()

    current_epoch = epoch + 1
    if current_epoch % CKPT_EVERY_DIFF == 0 or epoch == EPOCHS_DIFF - 1:

        save_ckpt('patchdiff', SLUG_DIFF, current_epoch,
                  unet=unet_diff.state_dict(), opt=opt_diff.state_dict())

        epoch_dir = FINAL_DIFF_BASE / f'eval_ep{current_epoch:04d}'
        epoch_dir.mkdir(exist_ok=True)
        print(f'\n[Inference] ep{current_epoch} — generowanie 1000 obrazów...')
        for _ in range(0, 1000, 16):
            save_imgs(ddim_sample(unet_diff, batch_size=16,
                                  n_steps=N_STEPS_DDIM).cpu(), str(epoch_dir))

        score(str(epoch_dir), 'patchdiff', f'{SLUG_DIFF}_ep{current_epoch:04d}')
        key = f'patchdiff__{SLUG_DIFF}_ep{current_epoch:04d}'
        fid = RESULTS.get(key, {}).get('FID', 'N/A')
        kid = RESULTS.get(key, {}).get('KID', 'N/A')
        avg_loss = epoch_loss / len(diff_loader)

        file_exists = csv_path_diff.exists()
        with open(csv_path_diff, 'a', newline='') as f:
            w = csv.writer(f)
            if not file_exists:
                w.writerow(['epoch', 'imgs_seen_M', 'loss', 'FID', 'KID'])
            w.writerow([
                current_epoch,
                round(current_epoch * len(dataset) / 1e6, 3),
                round(avg_loss, 5),
                fid, kid,
            ])
        print(f'  ep{current_epoch} | loss={avg_loss:.5f} | FID={fid}')
        unet_diff.train()
        torch.cuda.empty_cache()

# ── interpolacja latentna ─────────────────────────────────────────────
INTERP_DIR = FINAL_DIFF_BASE / 'interpolation'
INTERP_DIR.mkdir(exist_ok=True)

torch.manual_seed(42)
z1 = torch.randn(1, 3, IMG_SIZE, IMG_SIZE)
z2 = torch.randn(1, 3, IMG_SIZE, IMG_SIZE)
torch.save(z1, Path(DRIVE_DIR) / 'diff_v2_z1.pt')
torch.save(z2, Path(DRIVE_DIR) / 'diff_v2_z2.pt')

imgs_interp = [ddim_sample_from_z(unet_diff, (1-i/9)*z1 + (i/9)*z2)
               for i in range(10)]
utils.save_image(
    utils.make_grid(torch.cat(imgs_interp).clamp(-1,1), nrow=10, normalize=True),
    INTERP_DIR / 'interpolation.png'
)
utils.save_image((imgs_interp[0]+1)/2,  INTERP_DIR / 'img1.png')
utils.save_image((imgs_interp[-1]+1)/2, INTERP_DIR / 'img2.png')
print('Interpolacja →', INTERP_DIR / 'interpolation.png')

  [ckpt] no checkpoint for patchdiff (FULL_patch32_pmean-1.2_pfull0.5) — starting fresh


PatchDiffusion FULL:   0%|          | 0/1600 [00:00<?, ?it/s]

  [ckpt] patchdiff__FULL_patch32_pmean-1.2_pfull0.5__ep200.pt

[Inference] ep200 — generowanie 1000 obrazów...


/usr/local/lib/python3.12/dist-packages/cleanfid/fid.py:46: LinAlgWarning: Matrix is singular. The result might be inaccurate or the array might not have a square root.
  covmean, _ = linalg.sqrtm(sigma1.dot(sigma2), disp=False)


  [score] patchdiff__FULL_patch32_pmean-1.2_pfull0.5_ep0200: {'FID': np.float64(453.85), 'KID': 0.6209}
  ep200 | loss=0.44585 | FID=453.85
  [ckpt] patchdiff__FULL_patch32_pmean-1.2_pfull0.5__ep400.pt

[Inference] ep400 — generowanie 1000 obrazów...
  [score] patchdiff__FULL_patch32_pmean-1.2_pfull0.5_ep0400: {'FID': np.float64(436.38), 'KID': 0.5849}
  ep400 | loss=0.43556 | FID=436.38
  [ckpt] patchdiff__FULL_patch32_pmean-1.2_pfull0.5__ep600.pt

[Inference] ep600 — generowanie 1000 obrazów...
  [score] patchdiff__FULL_patch32_pmean-1.2_pfull0.5_ep0600: {'FID': np.float64(431.31), 'KID': 0.5763}
  ep600 | loss=0.42682 | FID=431.31
  [ckpt] patchdiff__FULL_patch32_pmean-1.2_pfull0.5__ep800.pt

[Inference] ep800 — generowanie 1000 obrazów...
  [score] patchdiff__FULL_patch32_pmean-1.2_pfull0.5_ep0800: {'FID': np.float64(435.87), 'KID': 0.5866}
  ep800 | loss=0.42657 | FID=435.87


KeyboardInterrupt: 

In [ ]:

BATCH_DDPM      = 64
LR_DDPM         = 2e-4
N_STEPS_DDIM    = 50


EPOCHS_DDPM     = 800
CKPT_EVERY_DDPM = 10
layers_per_block=1


SLUG_DDPM       = 'FULL_vanillaDDPM_cats'
FINAL_DDPM_BASE = Path(DRIVE_DIR) / 'final_vanilla_ddpm'
FINAL_DDPM_BASE.mkdir(parents=True, exist_ok=True)
csv_path_ddpm   = FINAL_DDPM_BASE / 'metrics_history.csv'

ddpm_loader = DataLoader(dataset, batch_size=BATCH_DDPM, shuffle=True,
                         num_workers=2, pin_memory=True)

unet_ddpm = UNet2DModel(
    sample_size=IMG_SIZE,
    in_channels=3,
    out_channels=3,
    layers_per_block=1,
    block_out_channels=(64, 128, 128),
    down_block_types=("DownBlock2D", "AttnDownBlock2D", "AttnDownBlock2D"),
    up_block_types=("AttnUpBlock2D", "AttnUpBlock2D", "UpBlock2D"),
).to(DEVICE)

scheduler_ddpm = DDPMScheduler(
    num_train_timesteps=1000,
    beta_schedule='squaredcos_cap_v2',
)
opt_ddpm = optim.AdamW(unet_ddpm.parameters(), lr=LR_DDPM)

# ── DDIM sampler ──────────────────────────────────────────────────────
def ddim_sample_ddpm(unet, batch_size=16, n_steps=50):
    sampler = DDIMScheduler(num_train_timesteps=1000,
                            beta_schedule='squaredcos_cap_v2')
    sampler.set_timesteps(n_steps)
    xt = torch.randn(batch_size, 3, IMG_SIZE, IMG_SIZE, device=DEVICE)
    unet.eval()
    with torch.no_grad():
        for t_step in sampler.timesteps:
            pred = unet(xt, t_step.expand(batch_size).to(DEVICE)).sample
            xt   = sampler.step(pred, t_step, xt).prev_sample
    return xt.clamp(-1, 1)

def ddim_sample_from_z_ddpm(unet, z, n_steps=50):
    """Inferencja od konkretnego z — do interpolacji latentnej."""
    sampler = DDIMScheduler(num_train_timesteps=1000,
                            beta_schedule='squaredcos_cap_v2')
    sampler.set_timesteps(n_steps)
    xt = z.to(DEVICE)
    unet.eval()
    with torch.no_grad():
        for t_step in sampler.timesteps:
            pred = unet(xt, t_step.expand(xt.size(0)).to(DEVICE)).sample
            xt   = sampler.step(pred, t_step, xt).prev_sample
    return xt.clamp(-1, 1)

# ── resume ────────────────────────────────────────────────────────────
state, start_ep = load_ckpt('vanilla_ddpm', SLUG_DDPM)
if state:
    unet_ddpm.load_state_dict(state['unet'])
    opt_ddpm.load_state_dict(state['opt'])
    print(f'[resume] od epoki {start_ep}')

# ── trening ───────────────────────────────────────────────────────────
for epoch in tqdm(range(start_ep, EPOCHS_DDPM), desc='Vanilla DDPM'):
    unet_ddpm.train()
    epoch_loss = 0.0

    for imgs in ddpm_loader:
        imgs  = imgs.to(DEVICE)
        B     = imgs.size(0)
        t     = torch.randint(0, 1000, (B,), device=DEVICE)
        noise = torch.randn_like(imgs)
        noisy = scheduler_ddpm.add_noise(imgs, noise, t)
        pred  = unet_ddpm(noisy, t).sample
        loss  = nn.functional.mse_loss(pred, noise)
        opt_ddpm.zero_grad(); loss.backward(); opt_ddpm.step()
        epoch_loss += loss.item()

    current_epoch = epoch + 1
    if current_epoch % CKPT_EVERY_DDPM == 0 or epoch == EPOCHS_DDPM - 1:

        # A. Checkpoint
        save_ckpt('vanilla_ddpm', SLUG_DDPM, current_epoch,
                  unet=unet_ddpm.state_dict(), opt=opt_ddpm.state_dict())

        # B. Generowanie obrazów
        epoch_dir = FINAL_DDPM_BASE / f'eval_ep{current_epoch:04d}'
        epoch_dir.mkdir(exist_ok=True)
        print(f'\n[Inference] ep{current_epoch} — generowanie 1000 obrazów...')
        for _ in range(0, 1000, 16):
            save_imgs(ddim_sample_ddpm(unet_ddpm, batch_size=16,
                                       n_steps=N_STEPS_DDIM).cpu(), str(epoch_dir))

        # C. FID/KID
        score(str(epoch_dir), 'vanilla_ddpm', f'{SLUG_DDPM}_ep{current_epoch:04d}')
        key = f'vanilla_ddpm__{SLUG_DDPM}_ep{current_epoch:04d}'
        fid = RESULTS.get(key, {}).get('FID', 'N/A')
        kid = RESULTS.get(key, {}).get('KID', 'N/A')
        avg_loss = epoch_loss / len(ddpm_loader)

        # D. CSV
        file_exists = csv_path_ddpm.exists()
        with open(csv_path_ddpm, 'a', newline='') as f:
            w = csv.writer(f)
            if not file_exists:
                w.writerow(['epoch', 'imgs_seen_M', 'loss', 'FID', 'KID'])
            w.writerow([
                current_epoch,
                round(current_epoch * len(dataset) / 1e6, 3),
                round(avg_loss, 5),
                fid, kid,
            ])
        print(f'  ep{current_epoch} | loss={avg_loss:.5f} | FID={fid}')
        unet_ddpm.train()
        torch.cuda.empty_cache()

# ── interpolacja latentna ─────────────────────────────────────────────
INTERP_DDPM = FINAL_DDPM_BASE / 'interpolation'
INTERP_DDPM.mkdir(exist_ok=True)

torch.manual_seed(42)
z1 = torch.randn(1, 3, IMG_SIZE, IMG_SIZE)
z2 = torch.randn(1, 3, IMG_SIZE, IMG_SIZE)
torch.save(z1, Path(DRIVE_DIR) / 'ddpm_z1.pt')
torch.save(z2, Path(DRIVE_DIR) / 'ddpm_z2.pt')

imgs_interp = [ddim_sample_from_z_ddpm(unet_ddpm, (1-i/9)*z1 + (i/9)*z2)
               for i in range(10)]
utils.save_image(
    utils.make_grid(torch.cat(imgs_interp).clamp(-1,1), nrow=10, normalize=True),
    INTERP_DDPM / 'interpolation.png'
)
utils.save_image((imgs_interp[0]+1)/2,  INTERP_DDPM / 'img1.png')
utils.save_image((imgs_interp[-1]+1)/2, INTERP_DDPM / 'img2.png')
print('Interpolacja →', INTERP_DDPM / 'interpolation.png')

  [ckpt] resumed vanilla_ddpm__FULL_vanillaDDPM_cats__ep390.pt (ep 390)
[resume] od epoki 390


Vanilla DDPM:   0%|          | 0/410 [00:00<?, ?it/s]

  [ckpt] vanilla_ddpm__FULL_vanillaDDPM_cats__ep400.pt

[Inference] ep400 — generowanie 1000 obrazów...
  [score] vanilla_ddpm__FULL_vanillaDDPM_cats_ep0400: {'FID': np.float64(126.78), 'KID': 0.0672}
  ep400 | loss=0.03951 | FID=126.78
  [ckpt] vanilla_ddpm__FULL_vanillaDDPM_cats__ep410.pt

[Inference] ep410 — generowanie 1000 obrazów...
  [score] vanilla_ddpm__FULL_vanillaDDPM_cats_ep0410: {'FID': np.float64(102.72), 'KID': 0.0431}
  ep410 | loss=0.03826 | FID=102.72
  [ckpt] vanilla_ddpm__FULL_vanillaDDPM_cats__ep420.pt

[Inference] ep420 — generowanie 1000 obrazów...
  [score] vanilla_ddpm__FULL_vanillaDDPM_cats_ep0420: {'FID': np.float64(103.61), 'KID': 0.0484}
  ep420 | loss=0.03878 | FID=103.61
  [ckpt] vanilla_ddpm__FULL_vanillaDDPM_cats__ep430.pt

[Inference] ep430 — generowanie 1000 obrazów...
  [score] vanilla_ddpm__FULL_vanillaDDPM_cats_ep0430: {'FID': np.float64(94.67), 'KID': 0.0379}
  ep430 | loss=0.03866 | FID=94.67
  [ckpt] vanilla_ddpm__FULL_vanillaDDPM_cats__ep440.pt

KeyboardInterrupt: 

In [ ]:
# latent interpolation

In [ ]:
INTERP_DDPM = FINAL_DDPM_BASE / 'interpolation'
INTERP_DDPM.mkdir(exist_ok=True)

# 1. Definicja poprawnej funkcji pomocniczej dla SLERP
def slerp(z1, z2, alpha):
    """
    Spherically interpolates between two tensors z1 and z2,
    safely flattening them for high-dimensional geometric operations.
    """
    orig_shape = z1.shape

    z1_flat = z1.view(-1)
    z2_flat = z2.view(-1)

    z1_norm = z1_flat / torch.norm(z1_flat)
    z2_norm = z2_flat / torch.norm(z2_flat)

    cos_theta = torch.sum(z1_norm * z2_norm)
    cos_theta = torch.clamp(cos_theta, -1.0, 1.0)
    theta = torch.acos(cos_theta)

    if torch.sin(theta) < 1e-5:
        res = (1.0 - alpha) * z1_flat + alpha * z2_flat
    else:
        sin_theta = torch.sin(theta)
        subset_1 = torch.sin((1.0 - alpha) * theta) / sin_theta
        subset_2 = torch.sin(alpha * theta) / sin_theta
        res = subset_1 * z1_flat + subset_2 * z2_flat

    return res.view(orig_shape)

# Liczba kroków w jednym pasku (2 skrajne + 8 pośrednich = 10)
steps = 10
alphas = [i / (steps - 1) for i in range(steps)]

# ZMIANA: Chcemy teraz wygenerować 10 różnych pasków (par) z nowymi kotami
NUM_PAIRS = 10
torch.manual_seed(42)  # stały seed bazowy zapewnia, że zestaw 10 par będzie reprodukowalny

for pair_idx in range(1, NUM_PAIRS + 1):
    print(f"\n--- Generowanie Pary nr {pair_idx}/{NUM_PAIRS} ---")

    # Pętla za każdym razem wylosuje zupełnie nowe punkty startowe i końcowe
    z1 = torch.randn(1, 3, IMG_SIZE, IMG_SIZE)
    z2 = torch.randn(1, 3, IMG_SIZE, IMG_SIZE)

    # Zapisujemy punkty ukryte danej pary
    torch.save(z1, INTERP_DDPM / f'ddpm_pair{pair_idx}_z1.pt')
    torch.save(z2, INTERP_DDPM / f'ddpm_pair{pair_idx}_z2.pt')

    # ── Wariant 1: Interpolacja Liniowa z Normalizacją (NLERP) ──
    print(f"   [NLERP] Para {pair_idx}...")
    imgs_lerp = []
    for alpha in alphas:
        z_lerp = (1.0 - alpha) * z1 + alpha * z2

        # Poprawka skali wariancji dla przestrzeni wysokowymiarowych
        target_norm = (1.0 - alpha) * torch.norm(z1) + alpha * torch.norm(z2)
        z_nlerp = z_lerp * (target_norm / torch.norm(z_lerp))

        img = ddim_sample_from_z_ddpm(unet_ddpm, z_nlerp, n_steps=N_STEPS_DDIM)
        imgs_lerp.append(img)

    utils.save_image(
        utils.make_grid(torch.cat(imgs_lerp).clamp(-1, 1), nrow=10, normalize=True),
        INTERP_DDPM / f'interpolation_linear_lerp_pair{pair_idx}.png'
    )

    # ── Wariant 2: Interpolacja Sferyczna (SLERP) ──
    print(f"   [SLERP] Para {pair_idx}...")
    imgs_slerp = []
    for alpha in alphas:
        z_slerp = slerp(z1, z2, alpha)
        img = ddim_sample_from_z_ddpm(unet_ddpm, z_slerp, n_steps=N_STEPS_DDIM)
        imgs_slerp.append(img)

    utils.save_image(
        utils.make_grid(torch.cat(imgs_slerp).clamp(-1, 1), nrow=10, normalize=True),
        INTERP_DDPM / f'interpolation_spherical_slerp_pair{pair_idx}.png'
    )

    # Zapisujemy obrazy skrajne (Kot A i Kot B) — Poprawiona ścieżka INTERP_DDPM
    utils.save_image((imgs_lerp[0] + 1) / 2,  INTERP_DDPM / f'pair{pair_idx}_img1_base.png')
    utils.save_image((imgs_lerp[-1] + 1) / 2, INTERP_DDPM / f'pair{pair_idx}_img2_base.png')

print(f'\nWszystkie {NUM_PAIRS} par zostało pomyślnie wygenerowane!')
print(f'Pliki znajdziesz w folderze: {INTERP_DDPM}')


--- Generowanie Pary nr 1/10 ---
   [NLERP] Para 1...
   [SLERP] Para 1...

--- Generowanie Pary nr 2/10 ---
   [NLERP] Para 2...
   [SLERP] Para 2...

--- Generowanie Pary nr 3/10 ---
   [NLERP] Para 3...
   [SLERP] Para 3...

--- Generowanie Pary nr 4/10 ---
   [NLERP] Para 4...
   [SLERP] Para 4...

--- Generowanie Pary nr 5/10 ---
   [NLERP] Para 5...
   [SLERP] Para 5...

--- Generowanie Pary nr 6/10 ---
   [NLERP] Para 6...
   [SLERP] Para 6...

--- Generowanie Pary nr 7/10 ---
   [NLERP] Para 7...
   [SLERP] Para 7...

--- Generowanie Pary nr 8/10 ---
   [NLERP] Para 8...
   [SLERP] Para 8...

--- Generowanie Pary nr 9/10 ---
   [NLERP] Para 9...
   [SLERP] Para 9...

--- Generowanie Pary nr 10/10 ---
   [NLERP] Para 10...
   [SLERP] Para 10...

Wszystkie 10 par zostało pomyślnie wygenerowane!
Pliki znajdziesz w folderze: /content/drive/MyDrive/diffusion_cats/final_vanilla_ddpm/interpolation


In [ ]:
# creation of images

In [ ]:
import os
from pathlib import Path
import torch
import torch.nn as nn
from diffusers import UNet2DModel, DDIMScheduler
from torchvision import utils

# ── 1. USTAWIENIA I HIPERPARAMETRY ────────────────────────────────────
DEVICE          = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
IMG_SIZE        = 64          # Rozmiar obrazu (64x64)
N_STEPS_DDIM    = 50          # Liczba kroków odszumiania samplerem DDIM
DRIVE_DIR       = '/content/drive/MyDrive/diffusion_cats'

# Wybierz rozmiar siatki:
GRID_SIZE       = 6        # 6 oznacza siatkę 6x6 (36 obrazków)
N_IMGS          = GRID_SIZE * GRID_SIZE

FINAL_DDPM_BASE = Path(DRIVE_DIR) / 'final_vanilla_ddpm'
FINAL_DDPM_BASE.mkdir(parents=True, exist_ok=True)

# Dokładna ścieżka do Twojego zapisanego checkpointu (uwzględniająca podfolder)
PATH_TO_WEIGHTS = Path(DRIVE_DIR) / 'vanilla_ddpm__FULL_vanillaDDPM_cats__ep470.pt'

# ── 2. INICJALIZACJA ARCHITEKTURY UNET ────────────────────────────────
print(f"Inicjalizacja UNet2DModel na urządzeniu: {DEVICE}")
unet_ddpm = UNet2DModel(
    sample_size=IMG_SIZE,
    in_channels=3,
    out_channels=3,
    layers_per_block=1,
    block_out_channels=(64, 128, 128),
    down_block_types=("DownBlock2D", "AttnDownBlock2D", "AttnDownBlock2D"),
    up_block_types=("AttnUpBlock2D", "AttnUpBlock2D", "UpBlock2D"),
).to(DEVICE)

# ── 3. MECHANIZM INTELIGENTNEGO WCZYTYWANIA WAG ───────────────────────
if not PATH_TO_WEIGHTS.exists():
    raise FileNotFoundError(f"Nie znaleziono pliku wag pod ścieżką: {PATH_TO_WEIGHTS}")

print(f"Wczytywanie wag z pliku: {PATH_TO_WEIGHTS}...")
checkpoint = torch.load(PATH_TO_WEIGHTS, map_location=DEVICE)

if isinstance(checkpoint, dict) and 'unet' in checkpoint:
    unet_ddpm.load_state_dict(checkpoint['unet'])
    print("[Sukces] Wczytano stan UNet ze słownika checkpointu.")
elif isinstance(checkpoint, dict) and 'state_dict' in checkpoint:
    unet_ddpm.load_state_dict(checkpoint['state_dict'])
    print("[Sukces] Wczytano stan ze słownika 'state_dict'.")
else:
    unet_ddpm.load_state_dict(checkpoint)
    print("[Sukces] Wczytano bezpośredni state_dict.")

unet_ddpm.eval()

# ── 4. SAMPLER DDIM BATCHED ───────────────────────────────────────────
def ddim_sample_batch_ddpm(unet, batch_size=36, n_steps=50):
    sampler = DDIMScheduler(
        num_train_timesteps=1000,
        beta_schedule='squaredcos_cap_v2'
    )
    sampler.set_timesteps(n_steps)

    xt = torch.randn(batch_size, 3, IMG_SIZE, IMG_SIZE, device=DEVICE)

    with torch.no_grad():
        for t_step in sampler.timesteps:
            t_input = t_step.expand(batch_size).to(DEVICE)
            pred = unet(xt, t_input).sample
            xt = sampler.step(pred, t_step, xt).prev_sample

    return xt.clamp(-1, 1)

# ── 5. GENEROWANIE I UKŁADANIE SIATKI BEZ RAMKI ───────────────────────
print(f"\nRozpoczynam generowanie {N_IMGS} obrazów (Siatka {GRID_SIZE}x{GRID_SIZE})...")

torch.manual_seed(43)
generated_imgs = ddim_sample_batch_ddpm(unet_ddpm, batch_size=N_IMGS, n_steps=N_STEPS_DDIM).cpu()

# Tworzenie siatki - KLUCZOWA ZMIANA: padding=0 usuwa marginesy
grid = utils.make_grid(
    generated_imgs,
    nrow=GRID_SIZE,
    padding=0,          # <--- Brak jakichkolwiek przerw między obrazkami
    normalize=True
)

# Zapis gotowej siatki na dysk
output_path = FINAL_DDPM_BASE / f'ddpm_report_grid_no_padding_{GRID_SIZE}x{GRID_SIZE}3.png'
utils.save_image(grid, output_path)

print('\n[Koniec] Siatka wygenerowana pomyślnie!')
print(f'Plik wynikowy bez ramek zapisano w: {output_path}')

Inicjalizacja UNet2DModel na urządzeniu: cuda
Wczytywanie wag z pliku: /content/drive/MyDrive/diffusion_cats/vanilla_ddpm__FULL_vanillaDDPM_cats__ep470.pt...
[Sukces] Wczytano stan UNet ze słownika checkpointu.

Rozpoczynam generowanie 36 obrazów (Siatka 6x6)...

[Koniec] Siatka wygenerowana pomyślnie!
Plik wynikowy bez ramek zapisano w: /content/drive/MyDrive/diffusion_cats/final_vanilla_ddpm/ddpm_report_grid_no_padding_6x63.png


In [ ]:
import os
from pathlib import Path
import torch
import torch.nn as nn
from diffusers import UNet2DModel, DDIMScheduler
from torchvision import utils
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

# Zadbanie o brak interfejsu graficznego dla matplotlib
import matplotlib
matplotlib.use('Agg')

# ── 0. ZMIENNA ŚCIEŻKI GŁÓWNEJ ────────────────────────────────────────
DRIVE_DIR = '/content/drive/MyDrive/diffusion_cats'

# ── 1. USTAWIENIA I ŚCIEŻKI ───────────────────────────────────────────
DEVICE          = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
IMG_SIZE        = 64
N_STEPS_DDIM    = 50
NUM_TRAJECTORIES = 5
NUM_REPORTS      = 5

FINAL_DDPM_BASE = Path(DRIVE_DIR) / 'final_vanilla_ddpm'
PROGRESS_DIR    = FINAL_DDPM_BASE / 'training_progression'
PROGRESS_DIR.mkdir(parents=True, exist_ok=True)

# ── TUTAJ WPISZ SWOJE EPOKI I ŚCIEŻKI DO PLIKÓW Z WAGAMI ────────────────
CHECKPOINTS_TO_LOAD = {
    'Epoch 50':  Path(DRIVE_DIR) / 'vanilla_ddpm__FULL_vanillaDDPM_cats__ep050.pt',
    'Epoch 100': Path(DRIVE_DIR) / 'vanilla_ddpm__FULL_vanillaDDPM_cats__ep100.pt',
    'Epoch 225': Path(DRIVE_DIR) / 'vanilla_ddpm__FULL_vanillaDDPM_cats__ep225.pt',
    'Epoch 360': Path(DRIVE_DIR) / 'vanilla_ddpm__FULL_vanillaDDPM_cats__ep360.pt',
    'Epoch 470': Path(DRIVE_DIR) / 'vanilla_ddpm__FULL_vanillaDDPM_cats__ep470.pt',
}

# ── 2. INICJALIZACJA ARCHITEKTURY UNET ────────────────────────────────
unet_ddpm = UNet2DModel(
    sample_size=IMG_SIZE,
    in_channels=3,
    out_channels=3,
    layers_per_block=1,
    block_out_channels=(64, 128, 128),
    down_block_types=("DownBlock2D", "AttnDownBlock2D", "AttnDownBlock2D"),
    up_block_types=("AttnUpBlock2D", "AttnUpBlock2D", "UpBlock2D"),
).to(DEVICE)

# ── 3. POPRAWIONY SAMPLER DDIM (Z ZABEZPIECZENIEM KONTRASTU) ──────────
def ddim_sample_from_z_ddpm(unet, z, n_steps=50):
    sampler = DDIMScheduler(
        num_train_timesteps=1000,
        beta_schedule='squaredcos_cap_v2',
        clip_sample=True,          # <--- POPRAWKA: Wymusza trzymanie się zakresu [-1, 1] na każdym kroku
        clip_sample_range=1.0      # Zabezpiecza przed powstawaniem wypranych, szarych tekstur
    )
    sampler.set_timesteps(n_steps)
    xt = z.to(DEVICE)

    with torch.no_grad():
        for t_step in sampler.timesteps:
            t_input = t_step.expand(xt.size(0)).to(DEVICE)
            pred = unet(xt, t_input).sample
            xt = sampler.step(pred, t_step, xt).prev_sample

    return xt.clamp(-1, 1)


# ── 4. GŁÓWNA PĘTLA GENERUJĄCA ────────────────────────────────────────
for run_idx in range(1, NUM_REPORTS + 1):
    print(f"\n========================================================")
    print(f"GENEROWANIE ZESTAWIENIA RAPORTU {run_idx}/{NUM_REPORTS}")
    print(f"========================================================")

    torch.manual_seed(42 + run_idx)
    fixed_z = torch.randn(NUM_TRAJECTORIES, 3, IMG_SIZE, IMG_SIZE)

    generated_grid_results = {}

    for label, weight_path in CHECKPOINTS_TO_LOAD.items():
        if not weight_path.exists():
            continue

        print(f"  -> Wczytywanie wag: {label}...")
        checkpoint = torch.load(weight_path, map_location=DEVICE)

        if isinstance(checkpoint, dict) and 'unet' in checkpoint:
            unet_ddpm.load_state_dict(checkpoint['unet'])
        elif isinstance(checkpoint, dict) and 'state_dict' in checkpoint:
            unet_ddpm.load_state_dict(checkpoint['state_dict'])
        else:
            unet_ddpm.load_state_dict(checkpoint)

        unet_ddpm.eval()

        batch_imgs = ddim_sample_from_z_ddpm(unet_ddpm, fixed_z, n_steps=N_STEPS_DDIM)

        # ── POPRAWKA B: INTELIGENTNE MAPOWANIE DANYCH DO WYKRESU ──────────
        processed_batch = []
        for i in range(NUM_TRAJECTORIES):
            img_np = batch_imgs[i].cpu().permute(1, 2, 0).numpy()

            # Dynamiczne rozciągnięcie histogramu (Min-Max) do pełnego pasma [0, 1]
            # Zapobiega to efektowi szarej mgły, jeśli model zwraca np. wartości bliskie 0.
            img_min, img_max = img_np.min(), img_np.max()
            if img_max - img_min > 1e-5:
                img_color = (img_np - img_min) / (img_max - img_min)
            else:
                img_color = (img_np + 1.0) / 2.0

            processed_batch.append(np.clip(img_color, 0.0, 1.0))

        generated_grid_results[label] = processed_batch

    # ── C. RENDEROWANIE MACIERZY ──────────────────────────────────────
    num_cols = len(generated_grid_results)

    if num_cols > 0:
        fig, axes = plt.subplots(NUM_TRAJECTORIES, num_cols, figsize=(num_cols * 2.5, NUM_TRAJECTORIES * 2.5))

        if NUM_TRAJECTORIES == 1:
            axes = axes.reshape(1, -1)
        if num_cols == 1:
            axes = axes.reshape(-1, 1)

        for col_idx, (epoch_label, batch_data) in enumerate(generated_grid_results.items()):
            for row_idx in range(NUM_TRAJECTORIES):
                ax = axes[row_idx, col_idx]
                ax.imshow(batch_data[row_idx])

                if row_idx == 0:
                    ax.set_title(epoch_label, fontsize=12, fontweight='bold', pad=12)

                ax.axis('off')

        plt.tight_layout()

        output_path = PROGRESS_DIR / f'ddpm_trajectory_progression_set{run_idx:02d}.png'
        plt.savefig(output_path, bbox_inches='tight', dpi=150, facecolor='white')
        plt.close()

        print(f"  [Sukces] Wygenerowano nasycony raport: {output_path.name}")

print('\n[Koniec] Generowanie zakończone. Zdjęcia powinny być teraz znacznie ostrzejsze i żywsze!')


GENEROWANIE ZESTAWIENIA RAPORTU 1/5
  -> Wczytywanie wag: Epoch 50...
  -> Wczytywanie wag: Epoch 100...
  -> Wczytywanie wag: Epoch 225...
  -> Wczytywanie wag: Epoch 360...
  -> Wczytywanie wag: Epoch 470...
  [Sukces] Wygenerowano nasycony raport: ddpm_trajectory_progression_set01.png

GENEROWANIE ZESTAWIENIA RAPORTU 2/5
  -> Wczytywanie wag: Epoch 50...
  -> Wczytywanie wag: Epoch 100...
  -> Wczytywanie wag: Epoch 225...
  -> Wczytywanie wag: Epoch 360...
  -> Wczytywanie wag: Epoch 470...
  [Sukces] Wygenerowano nasycony raport: ddpm_trajectory_progression_set02.png

GENEROWANIE ZESTAWIENIA RAPORTU 3/5
  -> Wczytywanie wag: Epoch 50...
  -> Wczytywanie wag: Epoch 100...
  -> Wczytywanie wag: Epoch 225...
  -> Wczytywanie wag: Epoch 360...
  -> Wczytywanie wag: Epoch 470...
  [Sukces] Wygenerowano nasycony raport: ddpm_trajectory_progression_set03.png

GENEROWANIE ZESTAWIENIA RAPORTU 4/5
  -> Wczytywanie wag: Epoch 50...
  -> Wczytywanie wag: Epoch 100...
  -> Wczytywanie wag: Ep